# sklearn Pipeline — Rock vs. Mine Classification
## Neural networks for energy applications — Day 1, Exercise 1

In this exercise we build a complete machine-learning pipeline using scikit-learn to classify sonar signals as either **Rock (R)** or **Mine/Metal (M)**.

The dataset (`sonar.all-data.csv`) contains 208 samples, each with 60 continuous features representing sonar signal energy at different frequency bands. The task is a binary classification problem.

By the end of this notebook you will:
- Load and explore a real-world sonar dataset
- Encode the labels with an explicit, printed mapping (Mine = 1, Rock = 0)
- Build sklearn `Pipeline` objects that combine preprocessing and modelling
- Compare Logistic Regression, Random Forest, and SVM classifiers with cross-validation on the training rows
- Tune the chosen model with grid search, then evaluate it **once** on the held-out test rows
- Visualize results with bar charts, box plots, and a confusion matrix

**Evaluation protocol.** The 42 test rows are set aside in section 5 and used only once, in section 12. Every choice before that (model family, hyperparameters) is made with 5-fold cross-validation on the 166 training rows.

---
### 1 — Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score,
    GridSearchCV,
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

np.random.seed(42)
print("Imports complete.")

---
### 2 — Load the Sonar Dataset

The CSV has no header row. There are 61 columns: the first 60 are continuous sonar features and the last column is the class label (`R` = Rock, `M` = Mine/Metal).

**Tasks:**
- Load `sonar.all-data.csv` with `pd.read_csv` (set `header=None`)
- Assign column names: `feature_0` through `feature_59` for the features, and `label` for the last column
- Print the shape and display the first few rows

In [ ]:
# YOUR CODE HERE
# Hint: feature_cols = [f"feature_{i}" for i in range(60)]
# df.columns = feature_cols + ["label"]


---
### 3 — Explore the Data

**Tasks:**
- Print the shape, data types, and check for missing values
- Use `.describe().T` to see summary statistics
- Plot the class distribution as a bar chart

In [ ]:
# YOUR CODE HERE
# Print shape, dtypes, missing values


In [ ]:
# YOUR CODE HERE
# Summary statistics with .describe().T


In [ ]:
# YOUR CODE HERE
# Plot class distribution bar chart


---
### 4 — Prepare the Data

Separate features from labels and encode the string labels with an **explicit mapping**: Mine (`M`) = 1, the positive class, and Rock (`R`) = 0. Every Sonar lab in this course uses this mapping (Lab 2 this afternoon and the PyTorch lab on Day 2). `LabelEncoder` would sort the labels alphabetically and give M = 0, R = 1, silently swapping the positive class, so we write the mapping out and print it: precision and recall depend on which class is positive.

**Tasks:**
- Extract `X` as the 60 feature columns (`.values`)
- Extract `y_raw` as the `label` column
- Define `LABEL_MAP = {"R": 0, "M": 1}` and `CLASS_NAMES = ["Rock", "Mine"]`, and map the labels to integers `y`
- Print the shapes, the label mapping, and the class counts

In [ ]:
# YOUR CODE HERE
# Hint: LABEL_MAP = {"R": 0, "M": 1}; y = df["label"].map(LABEL_MAP).to_numpy()

---
### 5 — Train / Test Split and Cross-Validation Folds

We use an 80/20 split with stratification to preserve class balance. We split the row numbers and then index the arrays (the same partition as splitting `X` and `y` directly), so that section 10 can put the training rows back into file order. The split is stratified on the `M`/`R` label column; Lab 2 uses the identical call, so both labs hold out the same 42 test echoes.

The **42 test rows are then set aside until section 12**. To compare and tune models we use one set of 5 shuffled, stratified folds on the 166 training rows, shared by every model, so that differences between scores come from the models and not from the folds.

**Tasks:**
- Split the row numbers: `idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42, stratify=y_raw)`, then index `X` and `y`
- Create the shared folds: `cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)`
- Print the sizes and class counts of both sets

In [ ]:
# YOUR CODE HERE
# Hint: X_train, X_test = X[idx_train], X[idx_test]

---
### 6 — Pipeline 1: StandardScaler + Logistic Regression

An sklearn `Pipeline` chains a sequence of transforms followed by a final estimator. Inside `cross_val_score` the whole pipeline is refitted on the training part of each fold, so the scaler never sees the validation fold (no data leakage).

We estimate each pipeline's accuracy with 5-fold cross-validation on the training rows, using the shared `cv` folds. The test rows are not used here.

**Tasks:**
- Create a `Pipeline` with steps: `("scaler", StandardScaler())` and `("clf", LogisticRegression(max_iter=1000, random_state=42))`
- Estimate its accuracy with `cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring="accuracy")`
- Print the mean ± standard deviation and the fold scores

In [ ]:
# YOUR CODE HERE
# Hint: pipe_lr = Pipeline([("scaler", StandardScaler()), ("clf", LogisticRegression(...))])
# scores_lr = cross_val_score(pipe_lr, X_train, y_train, cv=cv, scoring="accuracy")

---
### 7 — Pipeline 2: StandardScaler + Random Forest

**Tasks:**
- Create a `Pipeline` with `StandardScaler` and `RandomForestClassifier(n_estimators=100, random_state=42)`
- Estimate its accuracy with `cross_val_score` on the training rows, using the same `cv` folds
- Print the mean ± standard deviation and the fold scores

In [ ]:
# YOUR CODE HERE
# Hint: scores_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv, scoring="accuracy")

---
### 8 — Pipeline 3: StandardScaler + SVC

**Tasks:**
- Create a `Pipeline` with `StandardScaler` and `SVC(kernel="rbf", random_state=42)`
- Estimate its accuracy with `cross_val_score` on the training rows, using the same `cv` folds
- Print the mean ± standard deviation and the fold scores

In [ ]:
# YOUR CODE HERE
# Hint: scores_svc = cross_val_score(pipe_svc, X_train, y_train, cv=cv, scoring="accuracy")

---
### 9 — Compare All Models and Choose One

Compare the cross-validated accuracy of the three pipelines on the same folds, next to a majority-class baseline, and choose the model family to tune: the one with the highest mean CV accuracy.

**Why cross-validation and not the test set?** Choosing a model by its test accuracy spends the test set: once a score has influenced a decision, it no longer estimates performance on new data. With about 33 echoes per fold, one echo moves a fold score by 3 points, so compare the gaps between models with the fold-to-fold spread before calling one of them "best".

**Tasks:**
- Collect the model names, pipelines, and CV scores in lists
- Score a majority-class baseline, `DummyClassifier(strategy="most_frequent")`, on the same folds
- Plot the mean CV accuracy of each model with its standard deviation as error bars, and the baseline as a dashed line
- Choose the model with the highest mean CV accuracy (`best_idx`, `best_name`, `best_pipe`)

In [ ]:
# YOUR CODE HERE
# Hint: model_names = ["Logistic Regression", "Random Forest", "SVC"]
# cv_scores = [scores_lr, scores_rf, scores_svc]; best_idx = int(np.argmax([s.mean() for s in cv_scores]))

**Your interpretation:** How large is the gap between the two best models compared with their fold-to-fold standard deviation? Would you call one of them clearly better?

---
### 10 — Cross-Validation Comparison

A single train/test split can be misleading, and so can a single summary number. First we look at the fold-level scores behind the means from sections 6–8.

Then a **row-order check**. The file is not in random order: all rocks come first, then all mines, and within each class neighbouring rows may be related (the UCI documentation says the original returns were numbered in increasing aspect angle). We rerun the same pipelines on the training rows put back into **file order**, with 5 stratified folds **without shuffling**, so that each validation fold is a contiguous block of the file. This is a diagnostic only: it uses the training rows alone and does not change the choice made in section 9.

**Tasks:**
- Make a box plot of the fold scores of the three models (`ax.boxplot(..., tick_labels=...)`)
- Row-order check: reorder the training rows into file order with `np.argsort(idx_train)`, rerun `cross_val_score` for each pipeline with `StratifiedKFold(n_splits=5, shuffle=False)`, and compare with the shuffled folds

In [ ]:
# YOUR CODE HERE
# Box plot of the fold scores
# Hint: ax.boxplot(list_of_score_arrays, tick_labels=model_names, patch_artist=True)

In [ ]:
# YOUR CODE HERE
# Row-order check (diagnostic only; training rows only)
# Hint: file_order = np.argsort(idx_train)
#       X_train_file, y_train_file = X_train[file_order], y_train[file_order]
#       cv_file = StratifiedKFold(n_splits=5, shuffle=False)
#       s_file = cross_val_score(pipe, X_train_file, y_train_file, cv=cv_file, scoring="accuracy")

**Your interpretation:** How much do the scores change when the validation folds are contiguous blocks of the file? What could make neighbouring rows alike, and what is the equivalent in energy data?

---
### 11 — GridSearchCV on the Chosen Model

We tune the model family chosen in section 9. `GridSearchCV` runs cross-validation on the training rows for every setting in the grid, using the same shared folds. Because pipeline parameter names are prefixed with the step name (e.g. `clf__C`), we can tune the estimator inside the pipeline, and the scaler is refitted within every fold.

The best CV score is itself slightly optimistic: it is the maximum over all the settings evaluated on the same folds. The test rows are still untouched.

**Tasks:**
- Define a parameter grid for the chosen model (e.g. for SVC: `clf__C`, `clf__gamma`; for Random Forest: `clf__n_estimators`, `clf__max_depth`, `clf__min_samples_split`)
- Run `GridSearchCV` on the training rows with `cv=cv` and `scoring="accuracy"`
- Print the best CV score and the best parameters (no test score yet)

In [ ]:
# YOUR CODE HERE
# Hint: grid_search = GridSearchCV(best_pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
# grid_search.fit(X_train, y_train)   # training rows only

---
### 12 — Final Test Evaluation and Confusion Matrix

The choice is now locked: the model family (section 9) and its hyperparameters (section 11) were chosen with cross-validation on the training rows, and `GridSearchCV` has refitted the best setting on all 166 training rows. We evaluate it **once** on the 42 held-out test rows, next to the majority-class baseline, and plot its confusion matrix. Do not go back and change the model because of this score: that would turn the test set into a validation set.

Each test echo is worth 2.4 percentage points (1/42), so report the score together with its count.

**Tasks:**
- Take the refitted best model, `grid_search.best_estimator_`, and predict the test rows once
- Print the test accuracy with its count (n of 42), the majority-class baseline on the same rows, and the classification report (`target_names=CLASS_NAMES`)
- Plot two confusion matrices side by side: raw counts and normalized (`ConfusionMatrixDisplay`, `confusion_matrix(..., normalize="true")`)

In [ ]:
# YOUR CODE HERE
# Hint: final_model = grid_search.best_estimator_; y_pred_test = final_model.predict(X_test)

---
### Summary

In this exercise we:

1. Loaded and explored the UCI Sonar dataset (208 samples, 60 features).
2. Encoded the labels with an explicit, printed mapping (Mine = 1, Rock = 0) and split the data 80/20 with stratification.
3. Built three sklearn `Pipeline` objects, each combining `StandardScaler` with a different classifier.
4. Compared Logistic Regression, Random Forest, and SVC with 5-fold cross-validation on the training rows, next to a majority-class baseline, and chose one family.
5. Checked that unshuffled folds in file order give much lower scores: row order carries structure.
6. Applied `GridSearchCV` on the same folds to tune the chosen model's hyperparameters.
7. Evaluated the locked model once on the 42 test rows and visualized its confusion matrix.

**Key takeaway:** sklearn `Pipeline` objects are the recommended way to chain preprocessing and modelling steps. They prevent data leakage during cross-validation and grid search, and make the entire workflow reproducible with a single `.fit()` / `.predict()` call. Choose with cross-validation on the training rows, then touch the test set once.